In [1]:
import numpy as np

In [2]:
np.random.seed(42)

In [1]:
class LinearRegression:
    def __init__(self,lr=0.01,n_iter=1000):
        self.lr = lr
        self.n_iter = n_iter
        self.w = None
        self.b = None
        self.cost_history = []
    def fit(self,x,y):
        #initialize parameters
        m,n = x.shape
        self.initialize_parameters(n)
        for i in range(self.n_iter):
            # forward pass
            y_pred = self.forward_propagation(x)
            # calculate cost
            cost = self.compute_cost(y_pred,y)
            self.cost_history.append(cost)
            #back propagation
            dw,db = self.backward_propagation(x,y,y_pred)
            # update parameters
            self.update_parameters(dw,db)
            if i%100 == 0:
                print(f"Iteration:{i},   cost:{cost:.4f}")
            
            

    def initialize_parameters(self,n):
        self.w = np.random.rand(n) * 0.01 # shape (n,) # not (n,1) or (1,n), just 1D array
        self.b = 0
    def forward_propagation(self,x):
        y_pred = np.dot(x,self.w) + self.b
        return y_pred
    def compute_cost(self,y_pred,y):
        m = len(y_pred)
        cost = (1/(2*m))*np.sum((y_pred-y)**2)
        return cost
    def backward_propagation(self,x,y,y_pred):
        dw = (1/len(y))*np.dot(x.T,(y_pred-y)) #read notes for clarification, dw dimension should = w, then only w-dw possible, so take x.T
        db = (1/len(y))*np.sum(y_pred-y)
        return dw,db

    def update_parameters(self,dw,db):
        self.w = self.w - self.lr*dw
        self.b = self.b - self.lr*db

    def predict(self,x):
        return self.forward_propagation(x)
        

Standard practice, especially for deep learning, is to initialize weights from a standard normal distribution and scale them down (e.g., by 0.01 or by the square root of the number of features). This ensures the initial output of the sigmoid is close to 0.5, avoiding a high initial loss and huge gradients, which makes the model more stable from the start—a good practice for any system aiming for reliability.

In [5]:
np.sum(np.array([1,2,3]))

np.int64(6)

In [11]:
np.random.rand(3,1)

array([[0.08038091],
       [0.9708884 ],
       [0.23164785]])

In [16]:
n_samples = 100
x = 2* np.random.rand(n_samples,1)   # shape: (n_samples,1) # 2D matrix # random values b/w 0 and 2
y = 4 + 3 *x[:,0] + np.random.randn(n_samples)     # y = 4 + 3x + noise #y is 1D # x[:,0] gives 1 D (n_samples,)
# Reshape for our model
x_train = x
y_train = y

print(f"Data shape: X={x_train.shape}, y={y_train.shape}")
print(f"First 5 samples:")
for i in range(5):
    print(f"  X={x_train[i, 0]:.2f}, y={y_train[i]:.2f}")

Data shape: X=(100, 1), y=(100,)
First 5 samples:
  X=1.68, y=7.72
  X=0.17, y=4.84
  X=0.11, y=3.99
  X=1.52, y=10.11
  X=0.49, y=6.58


In [17]:
np.random.randn(4) # Return a sample (or samples) from the "standard normal" distribution. # size (4,) 1D

array([1.45953586, 0.35993822, 1.77574914, 0.3384481 ])

In [18]:
x = 2* np.random.rand(100,1)
x[:,0].shape

(100,)

In [23]:
model = LinearRegression(lr=0.1, n_iter=1000)
model.fit(x_train,y_train)

Iteration:0,   cost:19.4946
Iteration:100,   cost:0.4025
Iteration:200,   cost:0.3963
Iteration:300,   cost:0.3961
Iteration:400,   cost:0.3961
Iteration:500,   cost:0.3961
Iteration:600,   cost:0.3961
Iteration:700,   cost:0.3961
Iteration:800,   cost:0.3961
Iteration:900,   cost:0.3961


In [28]:
model.w  # True values were: slope=3, intercept=4

array([2.94407488])

In [29]:
model.b  

np.float64(4.050247780727483)

## compare with sklearn

In [39]:
from sklearn.linear_model import LinearRegression as Linear_reg
model1= Linear_reg()

In [40]:
model1.fit(x_train,y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [41]:
model1.coef_

array([2.94407486])

In [42]:
model.intercept_

np.float64(4.050247804842144)

### Multi-Feature Linear Regression

In [5]:
# Generate multi-feature data
n_samples = 200
n_features = 3

X_multi = np.random.randn(n_samples, n_features)
true_weights = np.array([2.5, -1.3, 0.7])
true_bias = 3.0
y_multi = np.dot(X_multi, true_weights) + true_bias + 0.1 * np.random.randn(n_samples)

# Train model
model_multi = LinearRegression(lr=0.1, n_iter=500)
model_multi.fit(X_multi, y_multi)

print("Multi-feature Linear Regression Results:")
print(f"Learned weights: {model_multi.w}")
print(f"True weights: {true_weights}")
print(f"Learned bias: {model_multi.b:.4f}")
print(f"True bias: {true_bias}")

Iteration:0,   cost:10.9621
Iteration:100,   cost:0.0049
Iteration:200,   cost:0.0049
Iteration:300,   cost:0.0049
Iteration:400,   cost:0.0049
Multi-feature Linear Regression Results:
Learned weights: [ 2.49859297 -1.28896584  0.69931822]
True weights: [ 2.5 -1.3  0.7]
Learned bias: 3.0028
True bias: 3.0


## RidgeRegression

In [14]:
class RidgeRegression(LinearRegression):
    def __init__(self, lr=0.01, n_iter=1000, lambda_reg=0.1):
        super().__init__(lr,n_iter)
        self.lambda_reg = lambda_reg
    def compute_cost(self,y_pred,y):
        mse_cost = super().compute_cost(y_pred,y)
        m = len(y_pred)
        # abb l2(RidgeRegression) term
        reg_cost = (self.lambda_reg/(2*m))* np.sum(self.w **2)
        return mse_cost+reg_cost
    def backward_propagation(self,x,y,y_pred):
        m = len(y_pred)
        dw = (1/len(y))*np.dot(x.T,(y_pred-y))
        # add l2
        dw += (self.lambda_reg/m) * self.w
        db = (1/len(y))*np.sum(y_pred-y)
        return dw,db
        

In [8]:
n_samples = 100
x = 2* np.random.rand(n_samples,1)   # shape: (n_samples,1) # 2D matrix # random values b/w 0 and 2
y = 4 + 3 *x[:,0] + np.random.randn(n_samples)     # y = 4 + 3x + noise #y is 1D # x[:,0] gives 1 D (n_samples,)
# Reshape for our model
x_train = x
y_train = y

print(f"Data shape: X={x_train.shape}, y={y_train.shape}")
print(f"First 5 samples:")
for i in range(5):
    print(f"  X={x_train[i, 0]:.2f}, y={y_train[i]:.2f}")

Data shape: X=(100, 1), y=(100,)
First 5 samples:
  X=0.14, y=2.28
  X=1.42, y=7.64
  X=1.09, y=8.56
  X=0.16, y=4.47
  X=0.92, y=5.75


In [15]:
model = RidgeRegression(lr=0.1, n_iter=1000,lambda_reg=0.5)
model.fit(x_train,y_train)

Iteration:0,   cost:24.5738
Iteration:100,   cost:0.4694
Iteration:200,   cost:0.4601
Iteration:300,   cost:0.4596
Iteration:400,   cost:0.4596
Iteration:500,   cost:0.4596
Iteration:600,   cost:0.4596
Iteration:700,   cost:0.4596
Iteration:800,   cost:0.4596
Iteration:900,   cost:0.4596
